# Practice 4 — Tools & Agents

Build LLM-powered agents that can call tools and reason step by step.

**Topics:**
- Tool definition
- Tool binding
- ReAct agent
- Agent executor
- Multi-step reasoning
- Handling tool errors

In [ ]:
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings("ignore")

load_dotenv(override=True)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

if os.environ.get("OPENAI_API_KEY"):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
elif os.environ.get("GROQ_API_KEY"):
    llm = ChatGroq(model="llama3-8b-8192", temperature=0.2)
else:
    raise ValueError("Set OPENAI_API_KEY or GROQ_API_KEY in .env")

print("LLM ready.")

## 1 — Define Custom Tools

A tool is a Python function + a description. The LLM reads the description to decide when to use it.

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    '''Multiply two numbers.'''
    return a * b

@tool
def get_word_length(word: str) -> int:
    '''Return the number of letters in a word.'''
    return len(word)

@tool
def current_year() -> int:
    '''Return the current year.'''
    from datetime import datetime
    return datetime.now().year

print(multiply.name, multiply.description)
print(get_word_length.name, get_word_length.description)
print(current_year.name, current_year.description)

## 2 — Bind Tools to an LLM

The LLM can now decide to call any of the bound tools.

In [ ]:
llm_with_tools = llm.bind_tools([multiply, get_word_length, current_year])

response = llm_with_tools.invoke("What is 12 times 15?")
print(response)
print("\nTool calls:", response.tool_calls)

## 3 — ReAct Agent

Use a pre-built ReAct agent that loops until it finds the answer.

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

prompt = hub.pull("hwchase17/react")

tools = [multiply, get_word_length, current_year]

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

result = agent_executor.invoke({
    "input": "How many letters are in the word 'artificial' and what is 7 times 8?"
})
print(result["output"])

## 4 — Add Web Search

Give the agent access to current information.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults(max_results=3)

tools_with_search = [multiply, get_word_length, current_year, search]

agent_with_search = create_react_agent(llm, tools_with_search, prompt)
agent_executor_search = AgentExecutor(
    agent=agent_with_search,
    tools=tools_with_search,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=6
)

result = agent_executor_search.invoke({
    "input": "Who is the current Prime Minister of India and how many letters are in their full name?"
})
print(result["output"])

## 5 — Manual Tool Calling Loop

Understand what happens under the hood by writing the loop yourself.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [HumanMessage(content="What is 6 times 9?")]

# First LLM call: it decides to call a tool
ai_msg = llm_with_tools.invoke(messages)
print("AI message:", ai_msg)

# Execute each tool call manually
for tool_call in ai_msg.tool_calls:
    selected_tool = {"multiply": multiply, "get_word_length": get_word_length, "current_year": current_year}[tool_call["name"]]
    tool_output = selected_tool.invoke(tool_call["args"])
    messages.append(ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"]))

# Second LLM call: it uses the tool result to answer
final_msg = llm_with_tools.invoke(messages)
print("\nFinal answer:", final_msg.content)

## Exercises

1. Add a `divide` tool and ask the agent to calculate `(100 / 5) * 3`.
2. Create a tool that fetches the current weather for a city (use a free API like wttr.in).
3. Build a multi-agent sketch: one agent calculates, another searches, and a third combines their outputs.
4. Make the agent fail gracefully when a tool raises an error.
5. Log every tool call the agent makes.